# Academic Research Notebook: HOG, KNN & PSO Face Recognition
## Smart Event Ticketing Biometric Authentication

**Methodological Reference:**
> Hardjianto, M., Aryasanti, A., & Hayati, P. (2026). *Attendance System with Face Recognition Using K-Nearest Neighbor, Particle Swarm Optimization, and Histogram of Oriented Gradients*. Journal of Embedded System Security and Intelligent Systems (JESSI), 7(4), 50-66.

### Pipeline Overview:
1. **Acquisition & Alignment:** Cropped Face ROI resized to $128 \times 128$ pixels.
2. **Grayscale Conversion:** $\text{Gray} = 0.2989R + 0.5870G + 0.1140B$.
3. **HOG Feature Extraction:** $8 \times 8$ cells, 9 orientation bins, $2 \times 2$ blocks $\rightarrow$ **8,100 numerical values**.
4. **KNN Classification:** Euclidean distance metric $d(x, y) = \sqrt{\sum (x_i - y_i)^2}$.
5. **PSO Optimization:** Parameter tuning for neighborhood parameter $k$ and rejection threshold $\tau$.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Ensure project root in python path
import sys
sys.path.append(os.path.abspath('../..'))

from app.cv.preprocessing import convert_to_grayscale, resize_face, normalize_lighting
from app.cv.hog_extractor import hog_extractor
from app.ml.knn_model import FaceKNNClassifier
from app.ml.pso_optimizer import ParticleSwarmOptimizer
from app.ml.threshold import ThresholdCalibrator

print("Modules successfully imported.")
print(f"HOG Expected Descriptor Length: {hog_extractor.expected_dim}")

## 1. Computer Vision: Preprocessing and HOG Extraction
We create a sample face pattern and pass it through the exact preprocessing pipeline.

In [ ]:
# Generate a test face image
face_sample = np.full((160, 160, 3), 180, dtype=np.uint8)
cv2.circle(face_sample, (80, 80), 55, (130, 130, 130), -1) # Head
cv2.circle(face_sample, (60, 70), 8, (30, 30, 30), -1)   # Left Eye
cv2.circle(face_sample, (100, 70), 8, (30, 30, 30), -1)  # Right Eye
cv2.ellipse(face_sample, (80, 110), (18, 8), 0, 0, 180, (40, 40, 40), -1) # Mouth

# 1. Grayscale Conversion (BT.601 formula)
gray = convert_to_grayscale(face_sample)
# 2. Resizing to 128x128
resized = resize_face(gray, target_size=128)
# 3. Contrast Equalization
equalized = normalize_lighting(resized)
# 4. HOG Feature Extraction
features, hog_vis = hog_extractor.extract_features(equalized, visualize=True)

assert len(features) == 8100, f"Expected 8100 features, got {len(features)}"
print(f"HOG Extraction Complete: {len(features)} dimensional vector extracted.")
print(f"Vector Stats: Min={features.min():.4f}, Max={features.max():.4f}, Mean={features.mean():.4f}")

## 2. Visualization of Intermediate Image Processing Stages

In [ ]:
grads = hog_extractor.compute_gradients_manual(equalized)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(cv2.cvtColor(face_sample, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Capture")
axes[0].axis('off')

axes[1].imshow(equalized, cmap='gray')
axes[1].set_title("128x128 Equalized")
axes[1].axis('off')

axes[2].imshow(grads["gx"], cmap='jet')
axes[2].set_title("Horizontal Gradient (Gx)")
axes[2].axis('off')

axes[3].imshow(grads["magnitude"], cmap='inferno')
axes[3].set_title("Magnitude M")
axes[3].axis('off')

axes[4].imshow(hog_vis, cmap='hot')
axes[4].set_title("HOG Visual Representation")
axes[4].axis('off')

plt.tight_layout()
plt.show()

## 3. Machine Learning: Baseline KNN vs PSO-KNN Optimization
Executing parameter search over particle swarm positions $x_i = [k, \tau]^T$.

In [ ]:
from research.experiments.run_experiment import generate_benchmark_feature_dataset, evaluate_model_pipeline

X_train, y_train, X_test, y_test, is_genuine = generate_benchmark_feature_dataset()
print(f"Training set: {len(X_train)} samples across 20 enrolled identities (9 poses each).")
print(f"Test set: {len(X_test)} probes (20 genuine, 7 unknown impostors).")

# 1. Model A: Baseline KNN (k=3, tau=0.65)
knn_base = FaceKNNClassifier(k=3)
knn_base.fit(X_train, y_train)
base_metrics, _ = evaluate_model_pipeline(knn_base.classifier, 0.65, X_test, y_test, is_genuine)
print(f"Model A (Baseline): Accuracy={base_metrics['accuracy']*100:.1f}%, FAR={base_metrics['far']*100:.1f}%, FRR={base_metrics['frr']*100:.1f}%")

# 2. Model B: PSO-KNN Optimization
pso = ParticleSwarmOptimizer(num_particles=8, max_iter=8, bounds=[(1.0, 7.0), (0.45, 0.75)], random_state=42)
val_idx = np.random.choice(len(X_train), size=40, replace=False)
pso_result = pso.optimize(X_train, y_train, X_train[val_idx], [y_train[i] for i in val_idx], X_test[20:])

opt_k = pso_result["optimized_k"]
opt_tau = pso_result["optimized_threshold"]
print(f"PSO Convergence Result: Optimal k={opt_k}, Optimal tau={opt_tau:.4f}")

knn_opt = FaceKNNClassifier(k=opt_k)
knn_opt.fit(X_train, y_train)
opt_metrics, _ = evaluate_model_pipeline(knn_opt.classifier, opt_tau, X_test, y_test, is_genuine)
print(f"Model B (PSO-KNN): Accuracy={opt_metrics['accuracy']*100:.2f}%, FAR={opt_metrics['far']*100:.1f}%, FRR={opt_metrics['frr']*100:.1f}%")

## 4. Convergence & Biometric Comparison Charts

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(pso_result['fitness_history']) + 1), pso_result['fitness_history'], 'o-', color='#3b82f6', lw=2)
plt.title("PSO Swarm Global Best Convergence")
plt.xlabel("Iteration")
plt.ylabel("Fitness Value")
plt.grid(True, linestyle='--', alpha=0.5)

plt.subplot(1, 2, 2)
bars = ['Accuracy', 'FAR', 'FRR']
x = np.arange(len(bars))
plt.bar(x - 0.17, [base_metrics['accuracy']*100, base_metrics['far']*100, base_metrics['frr']*100], 0.34, label='Baseline (k=3)', color='#94a3b8')
plt.bar(x + 0.17, [opt_metrics['accuracy']*100, opt_metrics['far']*100, opt_metrics['frr']*100], 0.34, label=f'PSO-KNN (k={opt_k})', color='#3b82f6')
plt.xticks(x, bars)
plt.ylabel("Percentage (%)")
plt.title("Model Comparison")
plt.legend()
plt.grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()